In [1]:
import pandas as pd

In [2]:
#df = pd.read_csv('gold_ml_price_features.csv')
df = pd.read_csv('cz_real_estate.csv')

In [3]:
df.head(5)

,title,price,currency,location,description_en,description_native,construction_of_building,condition,equipped,area_of_property,...,age,garage,elevator,balcony,parking,barrier_free_access,cellar,near_public_transport,terrace,transaction_type
0,Inzerát již není v nabídce Pronájem bytu 1+1 •...,8000.0,CZK,"sídliště 9. května, Nejdek - Nejdek, Karlovars...",NaN,Rádi bychom Vám představili tento byt k pronáj...,Cihla,Dobrý,Částečně,NaN,...,NaN,0,0,0,0,0,1,0,0,rent
1,Inzerát již není v nabídce Prodej bytu 6+1 • 2...,534000.0,EUR,Bozener Str 64 Ludwigshafen Gartenstadt Rheinl...,NaN,"Na prodej je balík, který zahrnuje prostorný b...",NaN,NaN,NaN,NaN,...,Nad 50 let,1,1,1,0,1,1,1,0,sale
2,Inzerát již není v nabídce Prodej domu 7+1 • 2...,749000.0,EUR,", Severní Porýní-Vestfálsko",NaN,Níže naleznete překlad textu označeného XML ta...,NaN,Po rekonstrukci,NaN,850.0,...,Nad 50 let,1,0,1,1,0,1,1,1,sale
3,Inzerát již není v nabídce Prodej bytu 2+1 • 3...,165000.0,EUR,Mendelsohnstraße 12 Augsburg Oberhausen Bayern...,NaN,Atraktivní 2-pokojový investiční byt a byt pro...,NaN,Před rekonstrukcí,NaN,NaN,...,Nad 50 let,0,0,0,0,0,1,1,0,sale
4,Inzerát již není v nabídce Prodej bytu 4+1 • 9...,389000.0,EUR,"Augsburger Straße 29b, , Bavorsko",NaN,"Popis bytu Königsbrunn, Augsburger Straße\n• O...",NaN,Po rekonstrukci,NaN,NaN,...,Nad 50 let,1,1,1,0,0,1,1,0,sale


In [4]:
def create_textual_representation(row):
    textual_representation = f""" Title: {row["title"]},
Price: {row["price"]} {row["currency"]},

Description_in_native: {row["description_native"]},

Description_in_english: {row["description_en"]}""" # Let's test these features for now
    return textual_representation

In [5]:
df['textual_representation'] = df.apply(create_textual_representation, axis=1)

In [6]:
print(df['textual_representation'].values[0])

 Title: Inzerát již není v nabídce Pronájem bytu 1+1 • 37 m² bez realitky sídliště 9. května, Nejdek - Nejdek, Karlovarský kraj,
Price: 8000.0 CZK,

Description_in_native: Rádi bychom Vám představili tento byt k pronájmu v městě Nejdek. Tento byt o dispozici 1+1 se nachází v cihlové budově a je v osobním vlastnictví. Byt je umístěn v prvním patře a nabízí užitnou plochu 37 m2, což je ideální pro jednotlivce nebo pár.
Byt je v dobrém stavu a je připraven k okamžitému nastěhování od 1. února 2026. Disponuje také sklepem o velikosti 6 m2, který poskytuje dostatek úložného prostoru pro Vaše věci. Za byt se platí nájem ve výši 8 000 Kč měsíčně. Poplatek za služby se hradí zvlášť a činí přibližně 2 600 Kč při bydlení jedné osoby. Elektřina a plyn budou převedeny na nájemce.
Byt je částečně zařízený. Postel je rozkládací a nabízí spací plochu o rozměrech 160 × 200 cm. Z vybavení je v koupelně k dispozici pračka, v kuchyni lednice, trouba a varná deska.
Vratná kauce činí 14 000 Kč. Nájemní sml

In [7]:
import faiss
import requests
import numpy as np
import httpx
import asyncio
from tqdm.asyncio import tqdm

dim = 768 #emb dim - nomic-embed-text compatible

index = faiss.IndexFlatL2(dim)

X = np.zeros((len(df['textual_representation']), dim), dtype = 'float32')

In [8]:
async def get_embedding(client, text):
    res = await client.post('http://localhost:11434/api/embeddings',
                            json={'model': 'nomic-embed-text',
                                  'prompt': text
                                 }
                           )
    data = res.json()
    if 'embedding' not in data:
        print(f"Bad response: {data}, text: {text[:100]}")
        return [0.0] * 768  # return zeros as fallback
    return data['embedding']

async def build_embeddings(texts, concurrency=100):
    sem = asyncio.Semaphore(concurrency)
    async def get_with_sem(client, text):
        async with sem:
            return await get_embedding(client, text)
    
    async with httpx.AsyncClient(timeout=120) as client:
        tasks = [get_with_sem(client, t) for t in texts]
        return await tqdm.gather(*tasks, desc="Embedding")

embeddings = await build_embeddings(df['textual_representation'].tolist())
X = np.array(embeddings, dtype='float32')
index.add(X)

Embedding:  18%|██████████████████████▎                                                                                                     | 1676/9337 [02:13<10:16, 12.42it/s]

Bad response: {'error': 'the input length exceeds the context length'}, text:  Title: Pronájem bytu 4+1 • 104 m² bez realitky Wolfgang-Knabe-Straße 6, , Braniborsko,
Price: 1655.


Embedding:  27%|█████████████████████████████████                                                                                           | 2489/9337 [03:20<07:07, 16.01it/s]

Bad response: {'error': 'the input length exceeds the context length'}, text:  Title: Prodej domu • 204 m² bez realitky , Durynsko,
Price: 383000.0 EUR,

Description_in_native: N


Embedding:  27%|█████████████████████████████████▌                                                                                          | 2530/9337 [03:24<10:20, 10.96it/s]

...

Embedding:  93%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊        | 8717/9337 [12:44<00:46, 13.42it/s]

Bad response: {'error': 'the input length exceeds the context length'}, text:  Title: Prodej domu • 232 m² bez realitky Eichstätt Landershofen Bayern 85072,
Price: 620000.0 EUR,



Embedding:  94%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊       | 8799/9337 [12:52<00:36, 14.60it/s]

Bad response: {'error': 'the input length exceeds the context length'}, text:  Title: Prodej domu 1+1 • 64 m² bez realitky SONNIGES BAUGRUNDSTÜCK EXKLUSIV NUR FÜR DANWOOD-HAUSBAU


Embedding: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9337/9337 [13:44<00:00, 11.33it/s]


In [9]:
X

array([[-0.11916139,  0.35879853, -2.2516818 , ..., -0.01533411,
        -1.0484278 , -0.49729982],
       [ 0.02693893,  0.24972092, -2.2525816 , ..., -0.47833142,
        -0.87518525, -0.896702  ],
       [ 0.06546106,  0.8490632 , -2.9293454 , ..., -0.50100785,
        -0.90181863, -1.0894728 ],
       ...,
       [-0.19772479,  0.06062134, -2.4827573 , ..., -0.38338915,
        -1.089279  , -0.62276345],
       [-0.11976001,  0.17897739, -2.618079  , ..., -0.63951653,
        -0.515102  , -0.86531913],
       [-0.5915039 ,  0.85962033, -2.661572  , ..., -0.23808046,
        -0.83203715, -0.8043839 ]], shape=(9337, 768), dtype=float32)

In [12]:
faiss.write_index(index, 'index')

### `index` is now a vector database full of embeddings
Lets load it

In [15]:
index = faiss.read_index('index')

### Bad response issua
TODO: investigate, maybe add truncation of text length